In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# Create synthetic price data
# -----------------------------
np.random.seed(0)

n = 2000
returns = np.random.normal(0, 0.01, n)
price = 100 * np.exp(np.cumsum(returns))

data = pd.DataFrame({
    "price": price
})

# -----------------------------
# Simple strategy definition
# -----------------------------
def generate_signals(df, short_window=20, long_window=100):
    df = df.copy()
    df["short_ma"] = df["price"].rolling(short_window).mean()
    df["long_ma"] = df["price"].rolling(long_window).mean()

    df["signal"] = 0
    df.loc[df["short_ma"] > df["long_ma"], "signal"] = 1
    df.loc[df["short_ma"] <= df["long_ma"], "signal"] = -1

    return df["signal"]

def compute_returns(df, signal):
    # shift signal to avoid lookahead bias
    strat_returns = signal.shift(1) * df["price"].pct_change()
    return strat_returns.fillna(0)

# -----------------------------
# Walk-forward testing setup
# -----------------------------
train_size = 500
test_size = 200
step_size = 50   # <--- note: smaller than test_size (overlapping evaluation)

results = []

start = train_size

while start + test_size < len(data):

    train_end = start
    test_end = start + test_size

    train_data = data.iloc[:train_end]
    test_data = data.iloc[train_end:test_end]

    # -----------------------------
    # "Training" step (parameter selection)
    # -----------------------------
    # In a real system you would optimize here.
    # We keep it fixed for simplicity.
    short_window = 20
    long_window = 100

    # -----------------------------
    # Apply strategy
    # -----------------------------
    full_data = pd.concat([train_data, test_data])

    signals = generate_signals(full_data, short_window, long_window)
    strat_returns = compute_returns(full_data, signals)

    test_returns = strat_returns.iloc[train_end:test_end]

    cumulative_return = (1 + test_returns).prod() - 1

    results.append(cumulative_return)

    # move forward (NOT necessarily by test_size)
    start += step_size

# -----------------------------
# Summary
# -----------------------------
results = np.array(results)

print("Walk-forward test results:")
print(f"Number of windows: {len(results)}")
print(f"Mean return: {results.mean():.4f}")
print(f"Std of returns: {results.std():.4f}")

Walk-forward test results:
Number of windows: 26
Mean return: -0.0249
Std of returns: 0.1771
